# Activation-Steering & Anti-Alignment — Experiment Console

One interactive environment over the **`refusal_geometry`** pipeline. Every stage is a single
cell that wraps the tested `asw` CLI/library — no need to read the repo. Edit the **Control
Panel** (Cell 3), then *Run All*.

**Pipeline:** download → extract `d_refuse` (C2) → anti-alignment map (C1) → fit condition (C4)
→ evaluate defenses → ablations → adversarial → **report** (regenerates every table & figure).

Each stage:
- is **idempotent** — re-running skips finished work (caches + prompt-level resume); set `FORCE=True` to recompute,
- streams **live logs** and shows **inline tables/plots**,
- writes artifacts under `/kaggle/working/refusal_geometry/{cache,data,results,report}` (persist across *Save Version*).

> **GPU:** uses `device_map="auto"`. ≤14B fits a single T4 (use `QUANT="int8"`/`"nf4"` for 14B);
> the 70B point is a paid A100 run, not this notebook.

## 1 · Clone / update the repository

In [ ]:
# Clones on first run, fast-forwards on later runs. Uses the GITHUB_TOKEN Kaggle secret.
import os
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "refusal_geometry"

if not REPO.exists():
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    os.system(f"git clone -q https://{token}@github.com/SLIMIHamda/refusal_geometry.git {REPO}")
else:
    os.system(f"git -C {REPO} pull --ff-only -q")

os.chdir(REPO)
print("cwd:", os.getcwd())
os.system("git -C . log --oneline -1")

## 2 · Environment

Kaggle ships `torch` (cu121) preinstalled — we keep it. We bump `transformers>=4.43` so
Qwen-2.5 loads, and add the CPU-side report deps. Then we confirm the GPU-free test spine is green.

In [ ]:
%pip install -q "transformers==4.44.2" "accelerate>=0.29.3" "bitsandbytes>=0.43.1" "pyyaml" "pandas" "pyarrow" "scipy" "scikit-learn" "matplotlib" "tabulate" "tqdm"
print("\nSpine tests (no GPU needed):")
!python -m pytest -q

## 3 · Control Panel — edit, then *Run All*

In [ ]:
# ============================ EXPERIMENT SETTINGS ============================
# Model under study — any file in configs/models/
MODEL_CONFIG = "configs/models/dolphin-llama3-8b.yaml"
QUANT        = None            # None | "int8" | "nf4"   (int8/nf4 for 14B on one T4)

# Geometry / steering band
STEER_LAYERS = None            # None = config's steer_layers; or e.g. [13, 14, 15, 16]
ALPHA        = 8.0             # steering strength for steered defenses

# Benchmarks
HARMFUL_BENCH    = "harmbench"     # harmful refusal eval
OVERREFUSE_BENCH = "xstest"        # benign / over-refusal eval
BENIGN_FOR_COND  = "orbench"       # benign negatives for the condition vector (over-refusal set)
PROMPT_LIMIT     = 100             # cap prompts/eval for speed; None = full split

# Defenses to evaluate (each is a Generator the harness scores identically)
DEFENSES = ["none", "system_prompt", "abliteration", "cast", "wrapper"]

# Ablation (alpha sweep)
ABLATE_ALPHAS = [2, 4, 8, 16]

# Diagnostics — why did the steered defenses land where they did? (scripts/diagnose_steering.py)
# One GPU pass, seven checks (D1–D7): operator routing + frozen α (D1), perturbation scale vs
# residual norm (D2), condition firing rate (D3), steered responses side by side (D4), an α-ladder
# on the current d_refuse (D5), the minimal-pair format-confound test cos(d_refuse, d_placebo) (D6),
# and the CORRECTED refusal-vs-compliance contrast direction with its own α-ladder (D7 — the one
# aligned with the goal of inducing refusal in an uncensored model). Reads the extract/geometry/
# condition caches; writes report/diagnose_steering.json. Nothing lands in runs.sqlite.
DIAG_LIMIT  = 20                  # prompts per generation condition
DIAG_ALPHAS = [8, 32, 128, 512]   # α-ladder for D5/D7
DIAG_QUANT  = "nf4"               # nf4 keeps 8B on one T4 (fast); None = bf16 (matches extraction, offloads)

# Adversarial (C5) — Tier-1; only runs when RUN_ATTACKS=True. Demo defaults are deliberately
# cheap (few behaviours, short GCG). For the paper, raise/drop these and let `asw attack` use the
# pre-registered config (configs/base.yaml -> attack:). budget = ATTACK_STEPS × search_width.
ATTACK_LIMIT = 3               # behaviours from the AdvBench eval split
ATTACK_STEPS = 40              # GCG steps per behaviour (paper: 250)

# Statistics
SEEDS = [0]                    # [0, 1, 2] for the paper; 1 seed for a fast pass

# Stage toggles — switch off expensive stages while iterating
RUN_EXTRACT   = True
RUN_GEOMETRY  = True
RUN_CONDITION = True
RUN_DEFENSES  = True
RUN_DIAGNOSE  = True           # D1–D7 diagnostic of the steered-defense result (GPU; ~minutes at nf4)
RUN_ABLATION  = True
RUN_ATTACKS   = False          # Tier-1 (slow, GPU); persisted GCG suite when True
RUN_REPORT    = True

FORCE = False                  # True = recompute even if a cache exists
DB    = "results/runs.sqlite"
# ============================================================================
print("Model :", MODEL_CONFIG, "| quant:", QUANT)
print("Eval  :", HARMFUL_BENCH, "/", OVERREFUSE_BENCH, "| limit:", PROMPT_LIMIT, "| seeds:", SEEDS)
print("Defs  :", DEFENSES)

## 4 · Helpers

Thin wrappers so each later cell is one call: `asw(...)` runs a CLI subcommand with live logging;
`load_runs(DB)` / the report tables turn the manifest into DataFrames; the `*_exists()` checks
drive idempotency by reusing the repo's own cache-path logic.

In [ ]:
import subprocess, time, sqlite3
import pandas as pd
from IPython.display import display, Markdown, Image
%matplotlib inline

from asw.config import load_config
from asw.harness.cli import _drefuse_path, _geometry_labels_path, _condition_path
from asw.report.load import load_runs
from asw.report import tables, figures


def sh(cmd, title=None):
    """Run a shell command, stream stdout, raise on non-zero exit."""
    if title:
        print(f"\n{'='*64}\n▶ {title}\n{'='*64}")
    print("$", cmd)
    t0 = time.time()
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.stdout:
        print(p.stdout)
    if p.returncode != 0:
        print(p.stderr)
        raise RuntimeError(f"command failed ({p.returncode})")
    print(f"✓ {time.time()-t0:.1f}s")
    return p.stdout


def asw(subcmd, **flags):
    """Build and run `python -m asw.harness.cli <subcmd>` from keyword flags."""
    parts = [f"python -m asw.harness.cli {subcmd}"]
    for k, v in flags.items():
        if v is None or v is False:
            continue
        flag = "--" + k.replace("_", "-")
        if v is True:
            parts.append(flag)
        elif isinstance(v, (list, tuple)):
            parts.append(flag + " " + " ".join(str(x) for x in v))
        else:
            parts.append(f"{flag} {v}")
    return sh(" ".join(parts), title=subcmd)


def cfg():
    return load_config(MODEL_CONFIG)


def drefuse_exists():   return _drefuse_path(cfg()).exists()
def geometry_exists():  return _geometry_labels_path(cfg()).exists()
def condition_exists(): return _condition_path(cfg()).exists()


def show_runs(n=12):
    """Most recent runs from the manifest (every paper number traces back here)."""
    df = load_runs(DB)
    cols = [c for c in ["experiment", "model_id", "defense", "seed", "status", "started_at"] if c in df.columns]
    return df.sort_values("started_at", ascending=False)[cols].head(n) if not df.empty else df

print("helpers ready ·  d_refuse:", drefuse_exists(), "· geometry:", geometry_exists(), "· condition:", condition_exists())

## 5 · Download benchmarks → `data/*.jsonl`  (idempotent)

In [ ]:
from pathlib import Path

needed = sorted({HARMFUL_BENCH, OVERREFUSE_BENCH, BENIGN_FOR_COND, "advbench"})
missing = [b for b in needed if not Path(f"data/{b}.jsonl").exists()]
if FORCE or missing:
    # 520 so AdvBench covers the projection split [300,500) (Item 4); other sets fetch what they have
    sh(f"python scripts/download_benchmarks.py --benchmarks {' '.join(needed)} --limit 520",
       title="download benchmarks")
else:
    print("all benchmark jsonl present — skip")
sh("ls -la data/*.jsonl")

## 6 · Extract the refusal direction `d_refuse`  (C2)

Behavioral-contrast difference-in-means (same prompts, native vs forced-refusal) per band layer.
Cached to `cache/drefuse/<model>.npz`; layer-consistency cosines are logged as a sanity check.

In [ ]:
if RUN_EXTRACT and (FORCE or not drefuse_exists()):
    asw("extract", config=MODEL_CONFIG, layers=STEER_LAYERS, quant=QUANT)
elif RUN_EXTRACT:
    print("d_refuse cache present — skip (FORCE=True to recompute)")
else:
    print("RUN_EXTRACT=False — skipped")

## 7 · Anti-alignment map  (C1)

Projects activations onto `d_refuse` per layer and classifies the geometry by 95% CI sign.
**Anti-aligned (`<y,d> < 0`) is the headline** — the property naive steering assumes away.
Heatmap + table shown inline.

In [ ]:
if RUN_GEOMETRY and (FORCE or not geometry_exists()):
    asw("geometry-map", config=MODEL_CONFIG, quant=QUANT)
elif RUN_GEOMETRY:
    print("geometry labels present — skip (FORCE=True to recompute)")

if RUN_GEOMETRY:
    g = tables.table_geometry(load_runs(DB))
    display(g)
    Path("report/figures").mkdir(parents=True, exist_ok=True)
    p = figures.fig_anti_alignment_map(g, "report/figures/anti_alignment_map.png")
    if p:
        display(Image(p))

## 8 · Fit + characterize the harmful-input detector  (C4)

A CAST-style detector (harmful vs benign difference-in-means at the mid-band layer) that gates
the wrapper, so benign prompts pass through untouched. Beyond train separation accuracy,
`fit-condition` now reports **held-out** ROC-AUC, TPR/FPR at the chosen τ, a threshold-sensitivity
sweep, and the detector's **FPR on XSTest** (the over-refusal firing rate) — surfaced in the
*Detector characterization* section of the report (Item 4).

In [ ]:
if RUN_CONDITION and (FORCE or not condition_exists()):
    asw("fit-condition", config=MODEL_CONFIG, benign=BENIGN_FOR_COND, quant=QUANT)
elif RUN_CONDITION:
    print("condition vector present — skip (FORCE=True to recompute)")
else:
    print("RUN_CONDITION=False — skipped")

## 9 · Evaluate defenses

Each defense is scored on the **harmful** and **over-refusal** benchmarks over `SEEDS`. Runs
resume at prompt granularity, so re-execution only fills gaps. The inline table uses the
**prompt-clustered** CI (resamples prompts, not seed×temp rows — Item 1), matching the report.

In [ ]:
if RUN_DEFENSES:
    from tqdm.auto import tqdm
    grid = [(d, b) for d in DEFENSES for b in (HARMFUL_BENCH, OVERREFUSE_BENCH)]
    for d, b in tqdm(grid, desc="defense × benchmark"):
        asw("eval", config=MODEL_CONFIG, benchmark=b, defense=d, alpha=ALPHA,
            quant=QUANT, limit=PROMPT_LIMIT, seeds=SEEDS)
    # results_dir -> prompt-clustered CI from the per-run parquet (Item 1), same as the report
    display(tables.table_refusal(load_runs(DB), results_dir="results"))
else:
    print("RUN_DEFENSES=False — skipped")

## 9½ · Diagnose the steered defenses  (`scripts/diagnose_steering.py`)

The table above shows every **steered** defense (`abliteration` / `cast` / `wrapper`) near zero
while `system_prompt` moves refusal sharply — so the null is specific to the *intervention*, not
the harness. This stage runs one GPU pass that separates the causes and, crucially, tests whether
`d_refuse` is a genuine refusal direction or an **extraction-format artifact**:

- **D1** which operator each band layer was routed to, and whether α was ever frozen on a tuning split;
- **D2** the perturbation scale — is `α·d̂` (unit vector) even visible against `‖h‖`? prints the *natural* α = `‖μ_refusal − μ_native‖`;
- **D3** does the condition mask ever fire (an all-False mask makes `wrapper` ≡ `none`);
- **D4** `none` / `abliteration` / `wrapper` responses side by side — "no effect" vs "garbage" vs "refused but unscored";
- **D5** an α-ladder on the **current** `d_refuse`;
- **D6** the decisive test: `cos(d_refuse, d_placebo)` where the placebo continuation flips one token (`cannot`→`can`) — high cosine ⇒ the direction is mostly format;
- **D7** the same α-ladder on the **corrected** `d_contrast = unit(μ[refusal cont] − μ[compliance cont])` — matched position/length, one-token stance flip. **This is the first cut at the actual goal: does a properly-formed direction induce refusal in the uncensored model?**

Verdicts stream inline (PASS/FAIL/INFO); the full read-out lands in `report/diagnose_steering.json`,
and `d_contrast` is cached to `cache/drefuse/d_contrast_diagnostic.npz` for the next iteration.

> Reads the extract / geometry-map / fit-condition caches (run those stages first). Nothing here
> writes to `runs.sqlite` — it is an instrument, not a logged experiment.

In [ ]:
if RUN_DIAGNOSE:
    import json

    cmd = (f"python scripts/diagnose_steering.py --config {MODEL_CONFIG} "
           f"--benchmark {HARMFUL_BENCH} --over-benchmark {OVERREFUSE_BENCH} "
           f"--limit {DIAG_LIMIT} --alpha {ALPHA} "
           f"--alphas {' '.join(str(a) for a in DIAG_ALPHAS)}"
           + (f" --quant {DIAG_QUANT}" if DIAG_QUANT else ""))
    sh(cmd, title="diagnose steered-defense null (D1–D7)")

    diag = json.loads(Path("report/diagnose_steering.json").read_text(encoding="utf-8"))

    # D6 — format-confound cosines. Low cos(d_refuse, d_placebo) ⇒ the direction is mostly
    # refusal stance; high ⇒ mostly extraction format (the one-token minimal pair isolates it).
    d6 = pd.DataFrame({
        "cos(d_refuse, d_placebo)":      diag["D6"]["cos_placebo"],
        "cos(d_refuse, d_natural_comply)": diag["D6"]["cos_natural_comply"],
        "cos(d_refuse, d_contrast)":     diag["D6"]["cos_contrast"],
    })
    d6.index.name = "layer"
    display(Markdown("**D6 · construct validity** — is `d_refuse` refusal, or format?"))
    display(d6)

    # D5 vs D7 — does the CORRECTED direction move refusal where the current one is flat?
    d5 = pd.DataFrame(diag["D5"]).set_index("alpha")[["refusal", "comply", "degenerate"]]
    d7 = pd.DataFrame(diag["D7"]).set_index("alpha")[["refusal", "comply", "degenerate"]]
    ladder = d5.join(d7, lsuffix="  ·current d_refuse", rsuffix="  ·corrected d_contrast")
    display(Markdown("**D5 vs D7 · α-ladder** — refusal rate under raw-add "
                     "(current `d_refuse` left, corrected `d_contrast` right)"))
    display(ladder)
    print("→ Read-out: if D6 cos_placebo is high AND D7 refusal climbs where D5 stays flat, the "
          "extraction contrast is the bug — fix asw/geometry/extract.py to pair refusal vs "
          "compliance continuations, then re-extract and re-run the defenses.")
else:
    print("RUN_DIAGNOSE=False — skipped")

## 10 · Ablation — steering strength `α`  (+ pre-registration)

Sweeps the wrapper's `α` on the harmful benchmark (safety/utility trade-off). With `--select-over`
it also **freezes the pre-registered α** — the argmax of (harmful-refusal − over-refusal), tuned on
the disjoint `orbench` set — to `cache/alpha/<model>.json`, so headline runs use a strength chosen
off the test set (Item 3); `eval` warns on any steered run whose α differs. Other axes:
`axis="layers"`, `"condition"`, and `"neutral_op"` (the 3-way project/raw_add/skip micro-ablation
for neutral layers, Item 5).

> For the paper, run the α selection **before** the headline eval (Cell 9) so the strength is
> genuinely frozen on the tuning split.

In [ ]:
if RUN_ABLATION:
    # --select-over freezes the pre-registered alpha (argmax harmful-refusal − over-refusal on the
    # disjoint orbench set) to cache/alpha/<model>.json (Item 3).
    asw("ablate", config=MODEL_CONFIG, benchmark=HARMFUL_BENCH, axis="alpha",
        alphas=ABLATE_ALPHAS, select_over=BENIGN_FOR_COND, quant=QUANT,
        limit=PROMPT_LIMIT, seeds=SEEDS)
    a = tables.table_ablation(load_runs(DB), "alpha")
    display(a)
    p = figures.fig_alpha_tradeoff(a, "report/figures/alpha_tradeoff.png")
    if p:
        display(Image(p))
    # optional: the neutral-layer routing micro-ablation (Item 5) —
    #   asw("ablate", config=MODEL_CONFIG, benchmark=HARMFUL_BENCH, axis="neutral_op",
    #       quant=QUANT, limit=PROMPT_LIMIT, seeds=SEEDS)
else:
    print("RUN_ABLATION=False — skipped")

## 11 · Adversarial robustness  (C5)

Every defense is attacked by **identical code**, so the *relative* ASR is the claim. This stage
runs the persisted GCG suite through the `asw attack` CLI (like every other stage → lands in
`runs.sqlite` → flows into the report), covering the two honest, adaptive variants (Item 6):

- **static GCG** on `none` vs `wrapper` — the baseline relative comparison;
- **GCG *through* the defense** (`gcg-adaptive`) — the wrapper's steering hooks are active across
  every forward/backward pass, so the suffix is optimised against the model the attacker faces;
- **detector-aware GCG** (`gcg-detector`) — a hinge penalty pushes the condition projection below
  the firing threshold τ so the wrapper *never engages*.

The headline C5 result is the **ASR-vs-budget** curve of the wrapper vs undefended.

> **Tier-1 (slow, GPU).** Off by default. When `RUN_ATTACKS=True` it uses the cheap demo knobs
> (`ATTACK_LIMIT`, `ATTACK_STEPS`); the paper run drops them for the pre-registered config
> (`configs/base.yaml → attack:`, see `docs/PRE_REGISTRATION_C5.md`). **Validate `run_gcg` first**
> (`scripts/validate_gcg.py`); until it passes, report only *relative* ASR, never absolute.

In [ ]:
if RUN_ATTACKS:
    # Tier-1: each call optimises a GCG suffix per behaviour (budget = ATTACK_STEPS × search_width),
    # so keep ATTACK_LIMIT/ATTACK_STEPS small here. Results persist to runs.sqlite and are picked up
    # by the report (Cell 12). Prereqs: extract + geometry-map + fit-condition have run.
    common = dict(config=MODEL_CONFIG, alpha=ALPHA, behaviors="advbench",
                  limit=ATTACK_LIMIT, n_steps=ATTACK_STEPS, search_width=64, topk=64,
                  budgets=[640, 1280, 2560], quant=QUANT)

    # static GCG: undefended vs wrapper, attacked by identical code (the relative comparison)
    asw("attack", defense="none",    attack="gcg", **common)
    asw("attack", defense="wrapper", attack="gcg", **common)
    # the HONEST numbers: through the defense, and detector-aware evasion of the wrapper
    asw("attack", defense="wrapper", attack="gcg-adaptive", **common)
    if condition_exists():
        asw("attack", defense="wrapper", attack="gcg-detector", **common)

    at = tables.table_attacks(load_runs(DB))
    display(at)
    p = figures.fig_asr_vs_budget(at, "report/figures/asr_vs_budget.png")
    if p:
        display(Image(p))
else:
    print("RUN_ATTACKS=False — skipped (Tier-1). Set True to run the persisted GCG suite "
          "(static + through-defense + detector-aware) into the manifest and report.")

## 12 · Report — regenerate every table & figure

`asw report` reads `runs.sqlite` and writes `report/REPORT.md` + `report/tables/*.csv` +
`report/figures/*.png`. Nothing in the paper is hand-transcribed.

In [ ]:
if RUN_REPORT:
    asw("report", config=MODEL_CONFIG, out="report")
    display(Markdown(Path("report/REPORT.md").read_text(encoding="utf-8")))
    for png in sorted(Path("report/figures").glob("*.png")):
        display(Image(str(png)))
else:
    print("RUN_REPORT=False — skipped")

## 13 · Run manifest (provenance)

In [ ]:
display(show_runs(20))

---
### Persist & resume
`cache/`, `data/`, `results/`, `report/` live under `/kaggle/working`, so **Save Version**
keeps them. On the next run, Cell 1 fast-forwards the repo and every stage resumes from its
cache — re-running is cheap. Flip a `RUN_*` toggle off to skip a stage, or `FORCE=True` to
recompute one. To study another model, point `MODEL_CONFIG` at a different `configs/models/*.yaml`.